# Building RAG Application with LangChain and Ollama

### 0 : Installing Required Libraries

In [27]:
!pip install langchain-community pypdf
!pip install -qU langchain-ollama
!pip install -qU chromadb langchain-chroma langchain


### 1 : Loading a PDF File

In [28]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Think-And-Grow-Rich_2011-06.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

253


### 2 : Chunking the document

In [29]:
# splitting the document into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)


### generating embeddings for the document chunks

In [30]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="llama3.1:latest")

### saving embeddings to Chroma vector store

In [31]:
from langchain_chroma import Chroma

persist_dir = "./chroma_db"
collection_name = "example_collection"

# Newer Chroma no longer requires constructing chromadb.Client with legacy Settings.
# Let the LangChain Chroma wrapper manage the client (it will create the collection if needed).
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=persist_dir,
)

In [47]:
ids = vector_store.add_documents(documents=all_splits)

In [9]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [25]:
from langchain_ollama import ChatOllama

# Initialize model with tool-calling capabilities
llm = ChatOllama(
    model="llama3.1:latest", 
    temperature=0,
    validate_model_on_init=True  
)

In [26]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a book. "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, tools, system_prompt=prompt)
query = (
     "burning desire as a driving force"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


================================ Human Message =================================

burning desire as a driving force
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (4150591f-def0-453f-9250-3ba55099316e)
 Call ID: 4150591f-def0-453f-9250-3ba55099316e
  Args:
    query: burning desire as a driving force in personal growth and motivation
================================= Tool Message =================================
Name: retrieve_context

Source: {'author': 'Brod', 'keywords': 'free, ebook, think and grow rich, make money online, home based business, opportunity', 'creator': 'Writer', 'producer': 'OpenOffice.org 3.2', 'total_pages': 253, 'page': 95, 'creationdate': '2011-06-06T18:53:08+01:00', 'start_index': 1582, 'subject': 'Free Digital Download PDF eBook Edition', 'source': 'Think-And-Grow-Rich_2011-06.pdf', 'page_label': '96', 'title': 'Think and Grow Rich by Napoleon Hill'}
Content: didn't you reach that decision a lon